# 🚀 Visual Regression AI Training — Google Colab

**步骤说明：**
1. 先把 `FYP(VISUAL)` 整个文件夹 Zip 并上传到 Google Drive
2. 按顺序运行每个 Cell
3. 训练完成后模型会自动保存回 Google Drive

**运行前请确认：** Runtime → Change runtime type → **T4 GPU**

## ✅ Step 0 — 确认 GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
else:
    print('⚠️  没有检测到 GPU，请检查 Runtime → Change runtime type → T4 GPU')

import os, zipfile, shutil
from pathlib import Path

# ⚠️ 修改这里：你的 zip 文件在 Google Drive 里的路径
ZIP_PATH = '/content/drive/MyDrive/FYP_VISUAL_colab.zip'
PROJECT_DIR = '/content/FYP_VISUAL'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'找不到: {ZIP_PATH}\n请先把 FYP_VISUAL_colab.zip 上传到 Google Drive 根目录')

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

print('解压中...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(PROJECT_DIR)
print(f'解压完成 → {PROJECT_DIR}')

# 找到 visual_regression 所在的目录
possible = list(Path(PROJECT_DIR).rglob('visual_regression/__init__.py'))
if not possible:
    raise RuntimeError('解压后找不到 visual_regression/ 目录，请检查 zip 文件结构')
WORK_DIR = str(possible[0].parent.parent)
os.chdir(WORK_DIR)
print(f'工作目录: {os.getcwd()}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

In [ ]:
import os, zipfile, shutil
from pathlib import Path

# ⚠️ 修改这里：你的 zip 文件在 Google Drive 里的路径
ZIP_PATH = '/content/drive/MyDrive/FYP(VISUAL).zip'
PROJECT_DIR = '/content/FYP_VISUAL'

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f'找不到: {ZIP_PATH}\n请先把 FYP(VISUAL).zip 上传到 Google Drive 根目录')

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

print('解压中...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(PROJECT_DIR)
print(f'解压完成 → {PROJECT_DIR}')

# 找到 visual_regression 所在的目录
possible = list(Path(PROJECT_DIR).rglob('visual_regression/__init__.py'))
if not possible:
    raise RuntimeError('解压后找不到 visual_regression/ 目录，请检查 zip 文件结构')
WORK_DIR = str(possible[0].parent.parent)
os.chdir(WORK_DIR)
print(f'工作目录: {os.getcwd()}')

## 📦 Step 2 — 安装依赖

In [ ]:
# Colab 自带 torch+CUDA，只需补装其他依赖
!pip install -q opencv-python scikit-image datasets huggingface_hub
print('依赖安装完成')

In [ ]:
# 验证所有 import 正常
import sys
sys.path.insert(0, os.getcwd())
from visual_regression.ai_training import train_model, build_synthetic_dataset, _extract_diff_crop
print('✅ 所有模块 import 正常')

## 🌐 Step 3 — 下载训练数据集

从 HuggingFace 下载 5000 张 UI 截图（约 1.5-2.5GB），需要 30-60 分钟

In [ ]:
from pathlib import Path

# 检查是否已有数据集（Drive 里缓存过）
DRIVE_DATASET_CACHE = '/content/drive/MyDrive/FYP_datasets'
LOCAL_DATASET_DIR = Path('scratch/datasets')
MANIFEST_PATH = Path('.visual-regression/datasets/public-ui-manifest.json')

if Path(DRIVE_DATASET_CACHE).exists() and len(list(Path(DRIVE_DATASET_CACHE).rglob('*.png'))) >= 1000:
    print(f'发现 Drive 缓存数据集，复制中...')
    shutil.copytree(DRIVE_DATASET_CACHE, str(LOCAL_DATASET_DIR), dirs_exist_ok=True)
    print(f'复制完成：{len(list(LOCAL_DATASET_DIR.rglob("*.png")))} 张图片')
else:
    print('未找到缓存，开始从 HuggingFace 下载...')
    print('⏳ 预计 30-60 分钟')

In [ ]:
# 下载数据集（如果上面没有缓存的话）
import subprocess
result = subprocess.run(
    [sys.executable, 'download_datasets.py'],
    capture_output=False,  # 实时显示进度
    text=True,
    cwd=os.getcwd()
)
if result.returncode != 0:
    print('⚠️  下载有部分失败，但训练会继续使用已下载的数据')

In [ ]:
# 把数据集缓存到 Drive（下次不用重新下载）
if LOCAL_DATASET_DIR.exists() and len(list(LOCAL_DATASET_DIR.rglob('*.png'))) > 100:
    print('备份数据集到 Google Drive...')
    shutil.copytree(str(LOCAL_DATASET_DIR), DRIVE_DATASET_CACHE, dirs_exist_ok=True)
    img_count = len(list(Path(DRIVE_DATASET_CACHE).rglob('*.png')))
    print(f'✅ 已备份 {img_count} 张图片到 Drive')

# 检查 manifest
if MANIFEST_PATH.exists():
    import json
    data = json.loads(MANIFEST_PATH.read_text())
    print(f'✅ Manifest 就绪: {data.get("total_images", "?")} 张图片')
else:
    print('⚠️  未找到 manifest，训练将使用内置合成图像')

## 🧠 Step 4 — 开始训练 AI

预计训练时间：**6-8 小时**（T4 GPU）

- 80 epochs
- batch_size 128
- 约 200 万合成样本

In [ ]:
import json, time
from pathlib import Path
from visual_regression.config import WorkspacePaths
from visual_regression.ai_training import train_model

paths = WorkspacePaths(Path(os.getcwd()))
paths.ensure()

MANIFEST = Path('.visual-regression/datasets/public-ui-manifest.json')
MODEL_OUT = Path('.visual-regression/models/visual_ai_colab.pt')
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)

print('=' * 55)
print('  Starting AI Training')
print(f'  Epochs       : 80')
print(f'  Batch size   : 128')
print(f'  Samples/img  : 24')
print(f'  Manifest     : {MANIFEST.exists()}')
print(f'  GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('=' * 55)

t0 = time.time()
metadata = train_model(
    paths=paths,
    model_path=MODEL_OUT,
    epochs=80,
    batch_size=128,
    learning_rate=1e-3,
    samples_per_image=24,
    pixel_threshold=20,
    min_region_area=120,
    pretrained_backbone=True,
    dataset_manifest_path=MANIFEST if MANIFEST.exists() else None,
    max_public_images=5000,
)
elapsed = time.time() - t0

print(f'\n✅ Training complete in {elapsed/3600:.1f} hours')
print(f'   Accuracy : {metadata["accuracy"]:.4f}')
print(f'   Samples  : {metadata["samples"]:,}')
print(f'   Model    : {MODEL_OUT}')

## 💾 Step 5 — 保存模型到 Google Drive

In [ ]:
DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/FYP_models')
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 保存模型文件
shutil.copy2(str(MODEL_OUT), str(DRIVE_MODEL_DIR / 'visual_ai_colab.pt'))

# 保存 metadata JSON
meta_src = MODEL_OUT.with_suffix('.json')
if meta_src.exists():
    shutil.copy2(str(meta_src), str(DRIVE_MODEL_DIR / 'visual_ai_colab.json'))

# 保存评估报告
eval_reports = list(Path('.visual-regression/reports').glob('ai-eval-*.json'))
for r in eval_reports:
    shutil.copy2(str(r), str(DRIVE_MODEL_DIR / r.name))

print(f'✅ 模型已保存到 Google Drive: {DRIVE_MODEL_DIR}')
print(f'   文件: visual_ai_colab.pt')
print(f'   大小: {(DRIVE_MODEL_DIR / "visual_ai_colab.pt").stat().st_size / 1024**2:.1f} MB')
print()
print('📥 下一步：下载 visual_ai_colab.pt 放到你本地的')
print('   .visual-regression/models/ 目录下')

## 📊 Step 6 — 查看训练结果（每类准确率）

In [ ]:
print(f'\n总体准确率: {metadata["accuracy"]:.2%}\n')
print(f'{"类别":<25} {"Precision":>10} {"Recall":>10} {"样本数":>8}')
print('-' * 58)
for cls in metadata['evaluation']['per_class']:
    flag = '⚠️' if cls['precision'] < 0.7 or cls['recall'] < 0.7 else '✅'
    print(f"{flag} {cls['label']:<23} {cls['precision']:>10.2%} {cls['recall']:>10.2%} {cls['support']:>8,}")